# CV8508 Lab 1: Seeing Faces in Things
### Do animal faces help a face classifier find faces in clouds, toast and tree bark?

**Format.** You have about one hour. Use a coding agent for the plumbing; **you** decide what to test, check every number, and write the answers to the open questions in your own words. Run the notebook top to bottom before you submit it (`lab1_<surname>.ipynb`).

| Min | What you do |
|---|---|
| 0-10 | Look at the data. Play the human baseline: can *you* spot the pareidolic faces? Write a prediction. |
| 10-40 | Train a face / not-face classifier on human faces, fine-tune it on animal faces, and score both on pareidolia (AUC). |
| 40-55 | Look at failures. Answer the open questions. |
| 55-60 | Restart and run all, check figures, submit. |

**Cells marked `TODO` are yours** (or your agent's). Everything else is provided. Anything an agent writes must be tested on a small input first.

---
## The experiment in one paragraph
Face pareidolia is seeing a face in an object. Hamilton et al. (ECCV 2024) collected **Faces in Things**, about 5,000 web images with a box around each pareidolic face. They found that a face detector trained only on human faces (WIDER FACE) finds few of them, that **fine-tuning the same detector on animal faces (AnimalWeb) roughly doubles** the score, and that fine-tuning on pareidolia itself helps most. They suggest that learning to detect animal faces may be part of why we see faces in things.

We pose a small version as **classification**. Each example is a 128x128 *crop*: label 1 = a face, label 0 = no face. We train on human, then animal faces, and test on crops from the Faces-in-Things **test split**: positives are the annotated pareidolic faces, negatives are crops taken **outside** every annotated box of the same images. The metric is **AUC**: the probability that a random pareidolic face gets a higher face-score than a random outside-the-box crop (0.5 = chance, 1.0 = perfect).

| | Hamilton et al. | This lab |
|---|---|---|
| Task | face *detection* (boxes) | face / not-face *crop classification* |
| Model | RetinaFace | ResNet-18 (ImageNet-pretrained) |
| Test | Faces-in-Things test split (about 1,500 images) | the same split, 927 images after filtering |
| Human faces | WIDER FACE (~13k images) | UTKFace crops (5,000 here) |
| Animal faces | AnimalWeb (~21k images) | AFHQ (cat, dog, wild; 2,100 here) |
| Metric | average precision | AUC (threshold-free) |

**Links to lectures.** Lecture 1: perception as inference, priors that fire on ambiguous input. Lecture 2: filters and receptive fields; a network's first layers are hand-built filters learned from data.


In [ ]:
# ---- setup (provided) -------------------------------------------------
import glob, os, time, copy, zipfile, urllib.request, random
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve

BASE_URL = "https://YOUR-GITHUB-USERNAME.github.io/cv8508-labs/lab1/data"   # instructor: set this once the zips are hosted
ZIPS = {"cv8508_lab1_human_faces.zip": "human_faces", "cv8508_lab1_animal_faces.zip": "animal_faces",
        "cv8508_lab1_backgrounds_objects.zip": "backgrounds/train/objects", "cv8508_lab1_backgrounds_web.zip": "backgrounds/train/web",
        "cv8508_lab1_fit_test.zip": "fit_test", "cv8508_lab1_fit_train.zip": "fit_train"}
DATA = Path("data"); DATA.mkdir(exist_ok=True)
for z, check in ZIPS.items():
    if not (DATA / check).exists():
        print("downloading", z)
        urllib.request.urlretrieve(f"{BASE_URL}/{z}", z)
        zipfile.ZipFile(z).extractall(DATA)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
EPOCHS_1, EPOCHS_2 = 3, 2      # stage 1 (human), stage 2 (animal / control)


## Part 0. The four datasets (provided)
Each zip is one dataset. Check the counts and look at the samples before you trust anything.

In [ ]:
def list_files(pattern):
    return sorted(glob.glob(str(DATA / pattern)))

human_tr  = list_files("human_faces/train/*.jpg");          human_val  = list_files("human_faces/val/*.jpg")
animal_tr = list_files("animal_faces/train/*/*.jpg");       animal_val = list_files("animal_faces/val/*/*.jpg")
bg_tr = list_files("backgrounds/train/objects/*.jpg") + list_files("backgrounds/train/web/*.jpg")
bg_val = list_files("backgrounds/val/objects/*.jpg") + list_files("backgrounds/val/web/*.jpg")
fit = pd.read_csv(DATA / "fit_test" / "fit_test.csv")
fit_train_faces = list_files("fit_train/face/*.jpg")      # optional: pareidolic faces from the TRAIN split (extension only)
fit_face = [str(DATA / "fit_test" / f) for f in fit.face_crop]
fit_non  = [str(DATA / "fit_test" / f) for f in fit.nonface_crop]

print(pd.DataFrame({"train": [len(human_tr), len(animal_tr), len(bg_tr), len(fit_face) + len(fit_non)],
                    "val": [len(human_val), len(animal_val), len(bg_val), 0]},
                   index=["human faces", "animal faces", "backgrounds", "FIT test (pos + neg)"]))

def show_grid(files, n=8, title="", scores=None, cols=8):
    n = min(n, len(files)); rows = (n + cols - 1) // cols
    fig, ax = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7 + 0.4), squeeze=False)
    for a in ax.ravel(): a.axis("off")
    for k in range(n):
        ax.ravel()[k].imshow(Image.open(files[k]))
        if scores is not None: ax.ravel()[k].set_title(f"{scores[k]:.2f}", fontsize=9)
    fig.suptitle(title, fontsize=11); plt.tight_layout(); plt.show()

rs = random.Random(1)
for name, files in [("human faces", human_tr), ("animal faces", animal_tr), ("backgrounds", bg_tr),
                    ("FIT test: pareidolic faces (label 1)", fit_face), ("FIT test: crops outside the boxes (label 0)", fit_non)]:
    show_grid(rs.sample(files, 8), title=name)


## Part 1. Look and predict (10 min)
**1a. Human baseline.** Below are 20 test crops (10 pareidolic faces, 10 outside-the-box crops, shuffled). Look at each one and fill `my_guesses` with 1 (face) or 0 (no face). Do not peek at the labels. This gives you a human accuracy to compare with the networks later.

*Your guesses should come from your own eyes, not from an agent.*

In [ ]:
# ---- human baseline quiz (provided) ----------------------------------
rq = random.Random(7)
rows = rq.sample(range(len(fit)), 20)
quiz = [(fit_face[i], 1) for i in rows[:10]] + [(fit_non[i], 0) for i in rows[10:]]
rq.shuffle(quiz)
fig, ax = plt.subplots(4, 5, figsize=(10, 8.4))
for k, (a, (f, _)) in enumerate(zip(ax.ravel(), quiz)):
    a.imshow(Image.open(f)); a.set_title(str(k), fontsize=14); a.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# TODO 1a: fill in your 20 guesses (1 = face, 0 = no face), one per image number above
my_guesses = [None] * 20


In [ ]:
assert None not in my_guesses and len(my_guesses) == 20, "fill in all 20 guesses first"
truth = [lab for _, lab in quiz]
acc = np.mean(np.array(truth) == np.array(my_guesses))
print(f"human baseline on 20 crops: accuracy = {acc:.2f}  (10 faces + 10 non-faces, so chance = 0.50)")


**1b. Prediction (write before you train).** In the cell below, write your prediction in 2-3 sentences:
(i) roughly what AUC do you expect from a classifier trained on **human faces only** when it sees pareidolia (0.5 = chance, 1.0 = perfect), (ii) will **fine-tuning on animal faces** help, hurt or do nothing, and why, (iii) what would be a fair control for (ii)?

**My prediction:** *(write here)*

## Part 2. Train and test (30 min)
**Helpers (provided).** Images are read into one `uint8` tensor, augmented with a random crop and flip, and fed to a ResNet-18 whose last layer outputs one logit (face vs not-face). `train_model` is a plain training loop. You will write the dataset builder and the evaluation.

In [ ]:
# ---- helpers (provided) ---------------------------------------------
IMG = 112
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def read_images(files, size=IMG):
    """list of image paths -> uint8 tensor (N, 3, size, size)"""
    arr = np.stack([np.array(Image.open(f).convert("RGB").resize((size, size), Image.BILINEAR)) for f in files])
    return torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous()

def augment(x):
    """random resized crop (60-100% of the area) + horizontal flip, per image"""
    out = torch.empty_like(x)
    for i in range(len(x)):
        s = int(IMG * np.sqrt(np.random.uniform(0.6, 1.0)))
        a, b = np.random.randint(0, IMG - s + 1), np.random.randint(0, IMG - s + 1)
        c = F.interpolate(x[i:i + 1, :, a:a + s, b:b + s].float(), size=(IMG, IMG), mode="bilinear", align_corners=False)
        if np.random.rand() < 0.5: c = c.flip(3)
        out[i] = c[0].round().clamp(0, 255).byte()
    return out

def prep(x):
    return ((x.float() / 255) - MEAN) / STD

def make_model(pretrained=True, seed=SEED):
    torch.manual_seed(seed)
    m = torchvision.models.resnet18(weights="IMAGENET1K_V1" if pretrained else None)
    m.fc = nn.Linear(512, 1)
    return m.to(device)

def train_model(model, X, y, epochs, lr, bs=64, seed=SEED):
    """binary cross-entropy training; X uint8 (N,3,H,W), y float32 (N,) with 1 = face"""
    rng = np.random.RandomState(seed); np.random.seed(seed); torch.manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * ((len(X) + bs - 1) // bs))
    for ep in range(epochs):
        model.train(); perm = rng.permutation(len(X)); tot = 0; t0 = time.time()
        for k in range(0, len(X), bs):
            idx = perm[k:k + bs]
            xb = prep(augment(X[idx])).to(device); yb = torch.from_numpy(y[idx]).float().to(device)
            loss = F.binary_cross_entropy_with_logits(model(xb).squeeze(1), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item() * len(idx)
        print(f"  epoch {ep + 1}/{epochs}  loss {tot / len(X):.3f}  ({time.time() - t0:.0f}s)")

@torch.no_grad()
def predict_scores(model, X, bs=128):
    """probability that each crop is a face"""
    model.eval()
    return torch.cat([torch.sigmoid(model(prep(X[k:k + bs]).to(device)).squeeze(1)).cpu() for k in range(0, len(X), bs)]).numpy()


**TODO 1: dataset builder.** Write `make_dataset(pos_files, neg_files)`: it returns `X` (uint8 tensor from `read_images`, positives first) and `y` (float32 array, 1 for positives, 0 for negatives).

*Agent prompt idea:* "Write make_dataset(pos_files, neg_files) using read_images; return X and a float32 label array; test it on 4 files."


In [ ]:
def make_dataset(pos_files, neg_files):
    # TODO 1
    raise NotImplementedError

X_demo, y_demo = make_dataset(human_tr[:3], bg_tr[:2])
print(X_demo.shape, X_demo.dtype, y_demo)       # expect torch.Size([5, 3, 112, 112]) torch.uint8 [1. 1. 1. 0. 0.]


**TODO 2: evaluation.** Write `evaluate(y_true, scores)` returning the **AUC** (`roc_auc_score`) of the scores against the true labels. AUC needs no threshold: it only asks whether faces get higher scores than non-faces.

In [ ]:
def evaluate(y_true, scores):
    # TODO 2
    raise NotImplementedError


In [ ]:
# sanity checks for TODO 2 (provided): perfect ranking -> 1.0, reversed -> 0.0, half-right -> 0.75
assert abs(evaluate(np.array([1, 1, 0, 0]), np.array([0.9, 0.8, 0.3, 0.2])) - 1.0) < 1e-9
assert abs(evaluate(np.array([1, 1, 0, 0]), np.array([0.1, 0.2, 0.8, 0.9])) - 0.0) < 1e-9
assert abs(evaluate(np.array([1, 1, 0, 0]), np.array([0.9, 0.3, 0.5, 0.2])) - 0.75) < 1e-9
print("evaluate() looks right")


**Data splits (provided).** Stage 1 uses 3,000 human faces and 3,000 backgrounds. Stage 2 uses 2,000 *different* animal faces and 2,000 backgrounds. The control uses 2,000 *different* human faces and 2,000 backgrounds, so animal and control get the same amount of extra training. All validation and test data are held out.

In [ ]:
rng = np.random.RandomState(SEED)
H = list(human_tr);  rng.shuffle(H)
A = list(animal_tr); rng.shuffle(A)
B = list(bg_tr);     rng.shuffle(B)
H1, H2 = H[:3000], H[3000:5000]          # stage 1 humans / control humans
A2 = A[:2000]                            # stage 2 animals
B1, B2, B3 = B[:3000], B[3000:5000], B[5000:7000]

# held-out sets
bg_v = list(bg_val); rng.shuffle(bg_v)
val_sets = {"human val": (human_val[:400], bg_v[:400]), "animal val": (animal_val[:400], bg_v[:400])}


In [ ]:
# FIT test: 1 pareidolic face crop (label 1) + 1 outside-the-box crop (label 0) per image
X_fit = torch.cat([read_images(fit_face), read_images(fit_non)])
y_fit = np.r_[np.ones(len(fit_face)), np.zeros(len(fit_non))].astype(int)
print("FIT test:", X_fit.shape, "faces:", y_fit.sum(), "non-faces:", (1 - y_fit).sum())


**TODO 3: stage 1, human faces only (model H).** Build the training set (`H1` faces vs `B1` backgrounds), create a pretrained model with `make_model`, train it with `train_model(..., epochs=EPOCHS_1, lr=3e-4)`, and evaluate it on (a) the FIT test crops, (b) the human validation set, (c) the animal validation set. Store the three AUCs in `results["H"]`.

Keep a copy of the weights (`copy.deepcopy(model.state_dict())`) as `state_H`: you will need to restart from H twice.

In [ ]:
results = {}

def full_eval(model, name):
    """evaluate on FIT test + both validation sets and store in results[name] (provided)"""
    r = {"FIT test": evaluate(y_fit, predict_scores(model, X_fit))}
    for vname, (pos, neg) in val_sets.items():
        Xv, yv = make_dataset(pos, neg)
        r[vname] = evaluate(yv, predict_scores(model, Xv))
    results[name] = r
    print(f"[{name}]  AUC   pareidolia (FIT test): {r['FIT test']:.3f}   human val: {r['human val']:.3f}   animal val: {r['animal val']:.3f}")

# TODO 3: stage 1 -- build X1, y1 from H1 / B1; make_model; train_model(..., epochs=EPOCHS_1, lr=3e-4);
#         full_eval(model, "H"); state_H = copy.deepcopy(model.state_dict())


**TODO 4: stage 2, fine-tune on animal faces (model H→A).** Continue training the *same* model on `A2` vs `B2` for `EPOCHS_2` epochs with a smaller learning rate (`1e-4`). Evaluate with `full_eval(model, "H->A")`.

In [ ]:
# TODO 4


**TODO 5: the control (model H→H2), then the results table.** Reload `state_H`, fine-tune on `H2` (800 *new human faces*) vs `B3` with exactly the same settings as TODO 4, and evaluate as `"H->H2"`. Then print a table with one row per model and three columns: AUC on the FIT test, on the human validation set and on the animal validation set.

In [ ]:
# TODO 5: control + results table


**TODO 6: look at the failures.** Reload the H→A weights into the model (`state_HA`), score the FIT test crops, and use `show_grid` to display: (a) the 16 face crops with the *lowest* scores (faces it misses), (b) the 16 non-face crops with the *highest* scores (false alarms). `show_grid(files, n, title, scores)` is provided; sort the files by score yourself.

In [ ]:
# TODO 6


## Part 3. Open questions (15 min)
Answer in a few sentences each, in your own words, pointing at your own table or figures. Label each claim as **evidence** (your data show it), **hypothesis** (plausible, untested) or **speculation**. There are no single right answers; a careful, specific answer beats a confident vague one.

1. **What does AUC tell you, and what does it hide?** A model can have a decent AUC and still be useless at its default threshold of 0.5. Pick one model, find the score threshold at which it would actually be useful on pareidolia, and say what you give up (false alarms on ordinary crops?) to get it. Why might a paper still prefer AUC or average precision to F1 at 0.5?
2. **Did animal faces help?** Compare H, H→A and the control H→H2. How big is the effect compared with what you would expect from a different random seed (you can estimate this by re-running with another `SEED`)? What claim does your evidence support, and what would you need to run to be more sure?
3. **What did the model learn "face" to mean?** Use your TODO 6 figures. What do the caught pareidolic faces have in common? The missed ones? The false alarms? Connect this to a first-layer filter or a receptive field from Lecture 2, or to a prior from Lecture 1.
4. **Where could the test itself mislead you?** Think about how the negatives were made (crops outside the annotated boxes), what the positives look like (crop around a box) and how both differ from the training data. Name two things that could inflate or deflate the scores.
5. **You versus the network.** Compare your 20-crop human baseline with the models. What kinds of examples do you and the model disagree on? What would a fair human-vs-model comparison need?
6. **Design the next experiment.** In one paragraph, propose a single experiment that would test whether *animal-face experience specifically* (and not just more or more varied data) explains the gain. State what result would change your mind.

*(Add one markdown cell per answer below.)*

**Q1.** *(write here)*

**Q2.** *(write here)*

**Q3.** *(write here)*

**Q4.** *(write here)*

**Q5.** *(write here)*

**Q6.** *(write here)*

## Extensions (only if you have time; pick one)
* **Which negatives?** Retrain stage 1 with `backgrounds/.../objects` only, then `web` only. Which kind of negative makes the model better at pareidolia, and why?
* **Seeds.** Repeat the whole pipeline with three seeds and report mean and standard deviation. Does the animal-vs-control gap survive?
* **Upper bound.** Fine-tune H on `fit_train_faces` (pareidolic faces from the train split) against web backgrounds. How far is the animal fine-tune from training on the thing itself?
* **Scratch vs pretrained.** Use `make_model(pretrained=False)`. Does ImageNet pretraining (which already contains animals) change the story?
* **Which pareidolia?** `fit_test.csv` has `person_or_creature` (Animal, Cartoon, Robot, Alien, Human, Other) and `hard_to_spot`. Compare AUC per category for H and H→A.
* **Coarse vs fine.** Blur the test crops (Gaussian, increasing sigma) and plot the score. Do faces survive low-pass filtering better than non-faces? Connect to spatial-frequency channels from Lecture 2.
* **Your own pareidolia.** Photograph five pareidolic things on campus, crop them to squares and score them. Which does the model like?